# Simulation: drogued drifter and point particle runs

Run drogued drifter and point particle simulations from observed
deployment positions. Six drifters (298--303) with 3 m drogues are
simulated in effective currents (Eulerian + Stokes) using the pipeline
from notebook 08. Surface and 3 m point particles provide baselines.

This notebook produces output files only -- no visualization.

## Imports

In [1]:
import shutil
from pathlib import Path

import copernicusmarine as cm
import numpy as np
import pandas as pd
import xarray as xr
from parcels import FieldSet, Particle, ParticleFile, ParticleSet, StatusCode, Variable
from parcels._core.statuscodes import FieldOutOfBoundError
from parcels.kernels import AdvectionRK4

from drogued_drifters.drifter import DroguedDrifter

/Users/wrath/src/github.com/geomar-od-lagrange/2025_drogued_drifters/.pixi/envs/default/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/var/folders/w1/m9mm9h9167z_gcfzfffr0rgsh6j6kj/T/ipykernel_9748/2363659501.py:8: UserWarning: This is an alpha version of Parcels v4. The API is not stable and may change without deprecation warnings.
  from parcels import FieldSet, Particle, ParticleFile, ParticleSet, StatusCode, Variable


## Parameters

In [2]:
LON_MIN = 9.0
LON_MAX = 13.0
LAT_MIN = 53.5
LAT_MAX = 56.0
TIME_START = "2023-04-24"
TIME_END = "2023-05-10"
DROGUE_DEPTH = 3.0
DT = 300.0
RUNTIME_HOURS = 288
CSV_PATH = "examples/baltic_drifters/data/drifters_clean.csv"
OUTPUT_DIR = "output"

## Load observed trajectories

In [3]:
df = pd.read_csv(CSV_PATH, parse_dates=["date_UTC"])
drifter_ids = sorted(df["D_number"].unique())

# Extract deployment position and time (first record per drifter)
deployments = {}
for d_num in drifter_ids:
    first = df[df["D_number"] == d_num].iloc[0]
    deployments[d_num] = {
        "lon": first["Longitude"],
        "lat": first["Latitude"],
        "time": first["date_UTC"],
    }

for d_num, dep in deployments.items():
    print(f"  {d_num}: ({dep['lat']:.4f}N, {dep['lon']:.4f}E) at {dep['time']}")

  298: (54.5117N, 10.2004E) at 2023-04-24 08:28:52
  299: (54.5124N, 10.2006E) at 2023-04-24 08:56:12
  300: (54.5093N, 10.1968E) at 2023-04-24 07:57:27
  301: (54.5101N, 10.1968E) at 2023-04-24 08:02:51
  302: (54.5095N, 10.1965E) at 2023-04-24 07:59:57
  303: (54.5096N, 10.1968E) at 2023-04-24 08:00:45


## Build effective current field

Same pipeline as notebook 08: load CMEMS physics and wave partitions,
compute depth-dependent Stokes drift from wave partitions, add to
Eulerian currents, add z=0 layer, build FieldSet.

In [4]:
LON = slice(LON_MIN, LON_MAX)
LAT = slice(LAT_MIN, LAT_MAX)
TIME = slice(TIME_START, TIME_END)

ds_phy = cm.open_dataset(
    dataset_id="cmems_mod_bal_phy_anfc_PT1H-i",
    service="arco-geo-series",
)[["uo", "vo"]].sel(
    longitude=LON, latitude=LAT, time=TIME, depth=slice(0, 5),
).load()

ds_phy

INFO - 2026-03-26T17:06:42Z - Selected dataset version: "202411"


INFO - 2026-03-26T17:06:42Z - Selected dataset part: "default"


<xarray.Dataset> Size: 350MB
Dimensions:    (time: 408, depth: 5, latitude: 150, longitude: 143)
Coordinates:
  * time       (time) datetime64[ns] 3kB 2023-04-24 ... 2023-05-10T23:00:00
  * depth      (depth) float32 20B 0.5016 1.516 2.548 3.602 4.684
  * latitude   (latitude) float32 600B 53.51 53.52 53.54 ... 55.96 55.97 55.99
  * longitude  (longitude) float32 572B 9.042 9.069 9.097 ... 12.93 12.96 12.99
Data variables:
    uo         (time, depth, latitude, longitude) float32 175MB nan nan ... nan
    vo         (time, depth, latitude, longitude) float32 175MB nan nan ... nan
Attributes: (12/19)
    Conventions:               CF-1.0
    comment:                   Data on cropped native product grid. Horizonta...
    compression:               yes
    contact:                   servicedesk.cmems@mercator-ocean.eu
    creation_date:             2024-11-25 17:05:11
    easternmost_longitude:     30.208656311035156
    ...                        ...
    southernmost_latitude:     53.008296966552734
    start_date:                2024-12-01 01:00:00
    stop_date:                 2024-12-01 12:00:00
    title:                     CMEMS NEMO hourly model fields
    westernmost_longitude:     9.041582107543945
    copernicusmarine_version:  2.3.0

In [5]:
WAVE_VARS = [
    "VHM0_WW", "VTM01_WW", "VMDR_WW",
    "VHM0_SW1", "VTM01_SW1", "VMDR_SW1",
    "VHM0_SW2", "VTM01_SW2", "VMDR_SW2",
]

ds_wav = cm.open_dataset(
    dataset_id="cmems_mod_bal_wav_anfc_PT1H-i",
    service="arco-geo-series",
).sel(longitude=LON, latitude=LAT, time=TIME)[WAVE_VARS].load()

ds_wav

INFO - 2026-03-26T17:08:05Z - Selected dataset version: "202311"


INFO - 2026-03-26T17:08:05Z - Selected dataset part: "default"


<xarray.Dataset> Size: 317MB
Dimensions:    (time: 408, latitude: 150, longitude: 144)
Coordinates:
  * time       (time) datetime64[ns] 3kB 2023-04-24 ... 2023-05-10T23:00:00
  * latitude   (latitude) float32 600B 53.51 53.52 53.54 ... 55.96 55.97 55.99
  * longitude  (longitude) float32 576B 9.014 9.042 9.069 ... 12.93 12.96 12.99
Data variables:
    VHM0_WW    (time, latitude, longitude) float32 35MB nan nan nan ... nan nan
    VTM01_WW   (time, latitude, longitude) float32 35MB nan nan nan ... nan nan
    VMDR_WW    (time, latitude, longitude) float32 35MB nan nan nan ... nan nan
    VHM0_SW1   (time, latitude, longitude) float32 35MB nan nan nan ... nan nan
    VTM01_SW1  (time, latitude, longitude) float32 35MB nan nan nan ... nan nan
    VMDR_SW1   (time, latitude, longitude) float32 35MB nan nan nan ... nan nan
    VHM0_SW2   (time, latitude, longitude) float32 35MB nan nan nan ... nan nan
    VTM01_SW2  (time, latitude, longitude) float32 35MB nan nan nan ... nan nan
    VMDR_SW2   (time, latitude, longitude) float32 35MB nan nan nan ... nan nan
Attributes:
    Conventions:               CF-1.0
    cmems_product_id:          BALTICSEA_ANALYSISFORECAST_WAV_003_010
    easternmost_longitude:     30.2080
    grid_resolution:           1 nautical mile (ie. 0.0167 degrees northward;...
    institution:               Baltic MFC, PU Finnish Meteorological Institute
    northernmost_latitude:     65.9081
    source:                    FMI-WAM_CMEMS
    southernmost_latitude:     53.0083
    title:                     CMEMS WAM model fields (hourly)
    westernmost_longitude:     9.0138
    copernicusmarine_version:  2.3.0

### Stokes drift profiles from wave partitions

Deep-water monochromatic approximation per partition, evaluated at each
Eulerian depth level including z=0.

In [6]:
g = 9.81
depth_levels = np.concatenate([[0.0], ds_phy.depth.values])

PARTITIONS = [
    ("VHM0_WW", "VTM01_WW", "VMDR_WW"),
    ("VHM0_SW1", "VTM01_SW1", "VMDR_SW1"),
    ("VHM0_SW2", "VTM01_SW2", "VMDR_SW2"),
]

u_stokes = xr.DataArray(
    np.zeros((len(ds_wav.time), len(depth_levels), len(ds_wav.latitude), len(ds_wav.longitude))),
    dims=["time", "depth", "latitude", "longitude"],
    coords={"time": ds_wav.time, "depth": depth_levels,
            "latitude": ds_wav.latitude, "longitude": ds_wav.longitude},
)
v_stokes = u_stokes.copy()

for hs_var, tp_var, dir_var in PARTITIONS:
    Hs = ds_wav[hs_var]
    T = ds_wav[tp_var]
    dir_from = ds_wav[dir_var]
    valid = Hs.notnull() & T.notnull() & dir_from.notnull() & (T > 0)
    A = Hs / 2.0
    sigma = 2 * np.pi / T
    k = sigma**2 / g
    theta = np.deg2rad(270.0 - dir_from)
    stokes_surf = A**2 * sigma * k

    for i, z in enumerate(depth_levels):
        decay = np.exp(-2 * k * float(z))
        u_stokes[dict(depth=i)] += (stokes_surf * decay * np.cos(theta)).where(valid, 0.0).fillna(0.0)
        v_stokes[dict(depth=i)] += (stokes_surf * decay * np.sin(theta)).where(valid, 0.0).fillna(0.0)

### Interpolate Stokes onto physics grid and build effective current

In [7]:
ds_stokes = xr.Dataset({"u_stokes": u_stokes, "v_stokes": v_stokes})

ds_stokes_phys = ds_stokes.interp(
    longitude=ds_phy.longitude,
    latitude=ds_phy.latitude,
    method="linear",
).fillna(0.0)

# Add z=0 layer to Eulerian by copying shallowest level
ds_phy_z0 = ds_phy.isel(depth=0).assign_coords(depth=0.0)
ds_phy_ext = xr.concat([ds_phy_z0, ds_phy], dim="depth")

# Apply land mask from Eulerian data
land_mask = ds_phy_ext["uo"].isnull()
ds_stokes_phys["u_stokes"] = ds_stokes_phys["u_stokes"].where(~land_mask)
ds_stokes_phys["v_stokes"] = ds_stokes_phys["v_stokes"].where(~land_mask)

# Effective current = Eulerian + Stokes
ds_eff = xr.Dataset({
    "U": (ds_phy_ext["uo"] + ds_stokes_phys["u_stokes"]).fillna(0.0),
    "V": (ds_phy_ext["vo"] + ds_stokes_phys["v_stokes"]).fillna(0.0),
})
ds_eff

<xarray.Dataset> Size: 840MB
Dimensions:    (depth: 6, latitude: 150, longitude: 143, time: 408)
Coordinates:
  * depth      (depth) float64 48B 0.0 0.5016 1.516 2.548 3.602 4.684
  * latitude   (latitude) float32 600B 53.51 53.52 53.54 ... 55.96 55.97 55.99
  * longitude  (longitude) float32 572B 9.042 9.069 9.097 ... 12.93 12.96 12.99
  * time       (time) datetime64[ns] 3kB 2023-04-24 ... 2023-05-10T23:00:00
Data variables:
    U          (time, latitude, longitude, depth) float64 420MB 0.0 0.0 ... 0.0
    V          (time, latitude, longitude, depth) float64 420MB 0.0 0.0 ... 0.0

### Build FieldSet

In [8]:
ds_eff["grid"] = xr.DataArray(
    data=0,
    attrs={
        "cf_role": "grid_topology",
        "topology_dimension": 2,
        "node_dimensions": "longitude latitude",
        "face_dimensions": (
            "longitude:longitude (padding: none) "
            "latitude:latitude (padding: none)"
        ),
        "vertical_dimensions": "depth:depth (padding: none)",
        "node_coordinates": "longitude latitude",
    },
)

fieldset = FieldSet.from_sgrid_conventions(ds_eff, mesh="spherical")
fieldset.add_constant("drogue_depth", DROGUE_DEPTH)

## Drifter kernel

Same kernel as notebook 08. Parcels returns velocities in deg/s on a
spherical mesh; the kernel converts to m/s for the Cartesian drifter
model and converts back afterwards.

In [9]:
DrifterParticle = Particle.add_variable(Variable("z_eff", dtype=np.float64, initial=0.0))

dd = DroguedDrifter()
_warm_state = {}

drifter_trajectory = {"lon": [], "lat": [], "time": [], "pid": []}

_DEG2M = 1852.0 * 60.0


def DroguedDrifterKernel(particles, fieldset):
    """Advect particles at the steady-state drift velocity of a buoy+drogue."""
    drogue_depth = fieldset.drogue_depth

    n = len(np.asarray(particles.lon))
    z_surface = np.zeros(n)
    z_drogue = np.full(n, drogue_depth)
    try:
        (u_b, v_b) = fieldset.UV[
            particles.time, z_surface, particles.lat, particles.lon, particles
        ]
        (u_d, v_d) = fieldset.UV[
            particles.time, z_drogue, particles.lat, particles.lon, particles
        ]
    except FieldOutOfBoundError:
        particles.state = StatusCode.Delete
        return

    u_b, v_b = np.asarray(u_b), np.asarray(v_b)
    u_d, v_d = np.asarray(u_d), np.asarray(v_d)
    dt = np.asarray(particles.dt)
    lon_arr = np.asarray(particles.lon)
    lat_arr = np.asarray(particles.lat)
    time_arr = np.asarray(particles.time)
    pid_arr = np.asarray(particles.trajectory)

    cos_lat = np.cos(np.deg2rad(lat_arr))
    deg2m_lon = _DEG2M * cos_lat
    u_b_ms = u_b * deg2m_lon
    v_b_ms = v_b * _DEG2M
    u_d_ms = u_d * deg2m_lon
    v_d_ms = v_d * _DEG2M

    n = len(u_b_ms)
    y0_warm = _warm_state.get("Y") if _warm_state.get("n") == n else None
    xd_drift_ms, yd_drift_ms, theta_final, Y_final = dd.get_final_drift_batch(
        U_b=u_b_ms, V_b=v_b_ms, U_d=u_d_ms, V_d=v_d_ms, y0=y0_warm,
    )
    _warm_state["Y"] = Y_final
    _warm_state["n"] = n
    z_eff = -dd.l * np.cos(theta_final)

    xd_drift = xd_drift_ms / deg2m_lon
    yd_drift = yd_drift_ms / _DEG2M

    particles.dlon += xd_drift * dt
    particles.dlat += yd_drift * dt
    particles.z_eff = z_eff

    drifter_trajectory["lon"].append(lon_arr.copy())
    drifter_trajectory["lat"].append(lat_arr.copy())
    drifter_trajectory["time"].append(time_arr.copy())
    drifter_trajectory["pid"].append(pid_arr.copy())


def DeleteOOB(particles, fieldset):
    """Convert out-of-bounds errors to Delete status."""
    state = np.asarray(particles.state)
    oob = (state == StatusCode.ErrorOutOfBounds) | (state == StatusCode.ErrorThroughSurface)
    if np.any(oob):
        particles.state = np.where(oob, StatusCode.Delete, state)

## Run drogued drifter simulation

Deploy 6 virtual drifters at the observed deployment positions and
times.

In [10]:
RUNTIME = RUNTIME_HOURS * 3600
SHALLOWEST_DEPTH = float(ds_eff.depth.values[0])
OUTPUTDT = 3600.0
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

# Prepare release arrays
release_lons = [deployments[did]["lon"] for did in drifter_ids]
release_lats = [deployments[did]["lat"] for did in drifter_ids]
release_times = [np.datetime64(deployments[did]["time"]) for did in drifter_ids]

# Convert release times to match FieldSet time coordinate
fieldset_time_origin = np.datetime64(ds_eff.time.values[0])

pset_drifter = ParticleSet(
    fieldset=fieldset,
    pclass=DrifterParticle,
    lon=release_lons,
    lat=release_lats,
    z=[SHALLOWEST_DEPTH] * len(drifter_ids),
    time=release_times,
)
pset_drifter.execute(
    kernels=[DroguedDrifterKernel, DeleteOOB],
    dt=DT,
    runtime=RUNTIME,
    verbose_progress=False,
)

In [11]:
# Assemble drogued drifter trajectories into a DataFrame
sim_drogued = {}
for lon_arr, lat_arr, time_arr, pid_arr in zip(
    drifter_trajectory["lon"],
    drifter_trajectory["lat"],
    drifter_trajectory["time"],
    drifter_trajectory["pid"],
):
    for j in range(len(pid_arr)):
        pid = int(pid_arr[j])
        if pid not in sim_drogued:
            sim_drogued[pid] = {"lon": [], "lat": [], "time": []}
        sim_drogued[pid]["lon"].append(lon_arr[j])
        sim_drogued[pid]["lat"].append(lat_arr[j])
        sim_drogued[pid]["time"].append(time_arr[j])

# Map trajectory index (0-5) to drifter IDs
pid_to_did = dict(zip(sorted(sim_drogued.keys()), drifter_ids))

rows = []
for pid, traj in sim_drogued.items():
    did = pid_to_did[pid]
    for lon, lat, t in zip(traj["lon"], traj["lat"], traj["time"]):
        rows.append({"D_number": did, "time": t, "lon": lon, "lat": lat})

df_drogued = pd.DataFrame(rows)
df_drogued.to_csv(output_dir / "sim_drogued_drifter.csv", index=False)
print(f"Saved {len(df_drogued)} rows to {output_dir / 'sim_drogued_drifter.csv'}")

Saved 20724 rows to output/sim_drogued_drifter.csv


## Run surface point particles

Surface (z=0) point particles with AdvectionRK4, same deployment
positions and times.

In [12]:
surface_store = str(output_dir / "sim_surface.zarr")
shutil.rmtree(surface_store, ignore_errors=True)

pset_surface = ParticleSet(
    fieldset=fieldset,
    pclass=Particle,
    lon=release_lons,
    lat=release_lats,
    z=[SHALLOWEST_DEPTH] * len(drifter_ids),
    time=release_times,
)
pset_surface.execute(
    kernels=[AdvectionRK4, DeleteOOB],
    dt=DT,
    runtime=RUNTIME,
    output_file=ParticleFile(store=surface_store, outputdt=OUTPUTDT),
    verbose_progress=False,
)

INFO: Output files are stored in /Users/wrath/src/github.com/geomar-od-lagrange/2025_drogued_drifters/output/sim_surface.zarr


/Users/wrath/src/github.com/geomar-od-lagrange/2025_drogued_drifters/.pixi/envs/default/lib/python3.14/site-packages/parcels/_core/particleset.py:415: ParticleSetWarning: Some of the particles have a start time difference that is not a multiple of outputdt. This could cause the first output of some of the particles that start later in the simulation to be at a different time than expected.
  _warn_outputdt_release_desync(outputdt, start_time, self._data["time"][:])
/Users/wrath/src/github.com/geomar-od-lagrange/2025_drogued_drifters/.pixi/envs/default/lib/python3.14/site-packages/parcels/_core/particlefile.py:281: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  np.less_equal(
/Users/wrath/src/github.com/geomar-od-lagrange/2025_drogued_drifters/.pixi/envs/default/lib/python3.14/site-packages/parcels/_core/particlefile.py:286: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is inte

## Run 3 m point particles

Point particles at drogue depth with AdvectionRK4, same deployment
positions and times.

In [13]:
DROGUE_DEPTH_LEVEL = float(ds_eff.depth.sel(depth=DROGUE_DEPTH, method="nearest"))

drogue_store = str(output_dir / "sim_3m.zarr")
shutil.rmtree(drogue_store, ignore_errors=True)

pset_drogue = ParticleSet(
    fieldset=fieldset,
    pclass=Particle,
    lon=release_lons,
    lat=release_lats,
    z=[DROGUE_DEPTH_LEVEL] * len(drifter_ids),
    time=release_times,
)
pset_drogue.execute(
    kernels=[AdvectionRK4, DeleteOOB],
    dt=DT,
    runtime=RUNTIME,
    output_file=ParticleFile(store=drogue_store, outputdt=OUTPUTDT),
    verbose_progress=False,
)

INFO: Output files are stored in /Users/wrath/src/github.com/geomar-od-lagrange/2025_drogued_drifters/output/sim_3m.zarr


## Save observed trajectories for convenience

Copy the clean observed CSV into the output directory so downstream
notebooks can find everything in one place.

In [14]:
obs_dst = output_dir / "obs_science.csv"
shutil.copy2(CSV_PATH, obs_dst)
print(f"Copied observed data to {obs_dst}")

Copied observed data to output/obs_science.csv


## Summary

In [15]:
# Drogued drifter summary
n_drifters = df_drogued["D_number"].nunique()
print(f"Drogued drifter: {n_drifters} drifters, {len(df_drogued)} total records")
for did in drifter_ids:
    sub = df_drogued[df_drogued["D_number"] == did]
    if len(sub) > 0:
        print(f"  D{did}: {len(sub)} steps, final ({sub.iloc[-1]['lat']:.4f}N, {sub.iloc[-1]['lon']:.4f}E)")

# Surface point particle summary
ds_s = xr.open_zarr(surface_store)
n_surf = ds_s.sizes["trajectory"]
valid_surf = np.isfinite(ds_s.lon.values).sum(axis=1)
print(f"\nSurface point particles: {n_surf} particles")
for i in range(n_surf):
    n_valid = valid_surf[i]
    last_valid = np.where(np.isfinite(ds_s.lon.values[i, :]))[0]
    if len(last_valid) > 0:
        idx = last_valid[-1]
        print(f"  Particle {i}: {n_valid} obs, final ({ds_s.lat.values[i, idx]:.4f}N, {ds_s.lon.values[i, idx]:.4f}E)")

# 3 m point particle summary
ds_d = xr.open_zarr(drogue_store)
n_drg = ds_d.sizes["trajectory"]
valid_drg = np.isfinite(ds_d.lon.values).sum(axis=1)
print(f"\n3 m point particles: {n_drg} particles")
for i in range(n_drg):
    n_valid = valid_drg[i]
    last_valid = np.where(np.isfinite(ds_d.lon.values[i, :]))[0]
    if len(last_valid) > 0:
        idx = last_valid[-1]
        print(f"  Particle {i}: {n_valid} obs, final ({ds_d.lat.values[i, idx]:.4f}N, {ds_d.lon.values[i, idx]:.4f}E)")

print(f"\nOutput files:")
print(f"  {output_dir / 'sim_drogued_drifter.csv'}")
print(f"  {surface_store}")
print(f"  {drogue_store}")
print(f"  {obs_dst}")

Drogued drifter: 6 drifters, 20724 total records
  D298: 3451 steps, final (54.4599N, 10.1648E)
  D299: 3446 steps, final (54.4600N, 10.1642E)
  D300: 3457 steps, final (54.4596N, 10.1678E)
  D301: 3456 steps, final (54.4587N, 10.1745E)
  D302: 3457 steps, final (54.4598N, 10.1663E)
  D303: 3457 steps, final (54.4611N, 10.1635E)

Surface point particles: 6 particles


  Particle 0: 289 obs, final (54.3252N, 10.8405E)


  Particle 1: 289 obs, final (54.3252N, 10.8404E)


  Particle 2: 288 obs, final (54.3252N, 10.8409E)


  Particle 3: 288 obs, final (54.3252N, 10.8404E)


...

  Particle 5: 288 obs, final (54.3252N, 10.8405E)



3 m point particles: 6 particles
  Particle 0: 289 obs, final (54.4457N, 10.1528E)


  Particle 1: 289 obs, final (54.4461N, 10.1528E)
  Particle 2: 288 obs, final (54.4519N, 10.1555E)


  Particle 3: 288 obs, final (54.4545N, 10.1649E)
  Particle 4: 288 obs, final (54.4473N, 10.1529E)


  Particle 5: 288 obs, final (54.4462N, 10.1528E)

Output files:
  output/sim_drogued_drifter.csv
  output/sim_surface.zarr
  output/sim_3m.zarr
  output/obs_science.csv
